# Tutorial 1: Working with Files in Python

### Kailyn Lau

This tutorial can be found in this link: [Working with Files in Python](https://realpython.com/working-with-files-in-python/). 

**Note:** Use Markdown headings to create sections in the notebook. Run each snippet of code in one cell.

In [ ]:
# initial imports
import os
from pathlib import Path
from datetime import datetime
import fnmatch
import glob
from tempfile import TemporaryFile, TemporaryDirectory
import shutil
import zipfile

## Reading and Writing Files using `with ... open as`

In [49]:
with open('data.txt', 'w') as f: # writing
    data = 'some data to be written to the file'
    f.write(data)
with open('data.txt', 'r') as f: # reading
    print(f.read())

some data to be written to the file


## Getting a Directory Listing using `os`, `scandir`, and `pathlib`
| Function      | Description |
| ----------- | ----------- |
| os.listdir()      | Returns a list of all files and folders in a directory       |
| os.scandir() (preferred) | Returns an iterator of all the objects in a directory including file attribute information        |
| pathlib.Path.iterdir() (preferred)| Returns an iterator of all the objects in a directory including file attribute information |

In [50]:
# using listdir
entries = os.listdir('.')
for entry in entries:
    print(entry)

2026
tutorial_1.ipynb
new_dir
new_dir_2
data.txt


In [51]:
# using scandir (preferred)
with os.scandir('../') as entries:
    for entry in entries:
        print(entry.name)

Tutorial 2 - JSON
LICENSE
Tutorial 1 - Files and Folders
Tutorial 3 - CSV
README.md
.git
.vscode


In [ ]:
# using pathlib (preferred)
# iterdir() also contains information about the file/dir like its name and file attributes
entries = Path('../../')
for entry in entries.iterdir():
    print(entry.name)

kdrama_pref_site
.DS_Store
cs315-week1-tasks


In [ ]:
# List all files in a directory using os.listdir (ignores nested dirs)
basepath = '../'
for entry in os.listdir(basepath):
    if os.path.isfile(os.path.join(basepath, entry)):
        print(entry)

# or with scandir()
basepath = '../../'
with os.scandir(basepath) as entries:
    for entry in entries:
        if entry.is_file():
            print(entry.name)

# # or with pathlib.Path()
# basepath = Path('my_directory/')
# files_in_basepath = basepath.iterdir()
# for item in files_in_basepath:
#     if item.is_file():
#         print(item.name)

# using pathlib.Path() and a generator expression
# basepath = Path('my_directory/')
# files_in_basepath = (entry for entry in basepath.iterdir() if entry.is_file())
# for item in files_in_basepath:
#     print(item.name)

LICENSE
README.md


In [55]:
# subdirectories

# List all subdirectories using os.listdir
basepath = '../../'
for entry in os.listdir(basepath):
    if os.path.isdir(os.path.join(basepath, entry)):
        print(entry)

# # List all subdirectories using scandir()
# basepath = 'my_directory/'
# with os.scandir(basepath) as entries:
#     for entry in entries:
#         if entry.is_dir():
#             print(entry.name)

# # List all subdirectory using pathlib
# basepath = Path('my_directory/')
# for entry in basepath.iterdir():
#     if entry.is_dir():
#         print(entry.name)

kdrama_pref_site
cs315-week1-tasks


### Getting File Attributes using `os.stat()`, `os.scandir()`, and `pathlib.Path()`

In [56]:
with os.scandir('./') as dir_contents:
    for entry in dir_contents:
        info = entry.stat()
        print(info.st_mtime)

# same thing but with pathlib
# current_dir = Path('my_directory')
# for path in current_dir.iterdir():
#     info = path.stat()
#     print(info.st_mtime)

1788546907.4668458
1788546908.490377
1788546907.4615552
1788546907.4617815
1788546917.7571251


In [57]:
# to actually see date and time:
def convert_date(timestamp):
    d = datetime.utcfromtimestamp(timestamp)
    formated_date = d.strftime('%d %b %Y')
    return formated_date

def get_files():
    dir_entries = os.scandir('./')
    for entry in dir_entries:
        if entry.is_file():
            info = entry.stat()
            print(f'{entry.name}\t Last Modified: {convert_date(info.st_mtime)}')
            
get_files()

tutorial_1.ipynb	 Last Modified: 04 Sep 2026
data.txt	 Last Modified: 04 Sep 2026


## Making Directories using `mkdir()` and `makedirs()`
| Function      | Description |
| ----------- | ----------- |
| os.mkdir()      | Creates a single subdirectory |
| pathlib.Path.mkdir()   | Creates single or multiple directories |
| os.makedirs() | Creates multiple directories, including intermediate directories |

In [58]:
# creating a single directory
os.mkdir('new_dir/') # might raise FileExistsError

p = Path('new_dir_2/')
p.mkdir() # might raise FileExistsError, use try/except to catch or change to `p.makedir(exist_ok=True)`

# p = Path('new_dir_2/')
# try:
#     p.mkdir()
# except FileExistsError as exc:
#     print(exc) 

FileExistsError: [Errno 17] File exists: 'new_dir/'

In [ ]:
# creating multiple directories

os.makedirs('2026/09/04') # 2016, subdir 09, subdir 04
# can directly pass permissions in - add mode=0o77o or whatever as an argument

# Or use
# p = Path('2026/09/04')
# p.mkdir(parents=True)

## Filename Pattern Matching

| Function      | Description |
| ----------- | ----------- |
| startswith()      | Tests if a string starts with a specified pattern and returns True or False |
| endswith() | Tests if a string ends with a specified pattern and returns True or False |
| fnmatch.fnmatch()   | Tests whether the filename matches the pattern and returns True or False  |
| glob.glob() | Returns a list of filenames that match a pattern |
| glob.iglob() | Returns an iterator of filenames that match a pattern | 
| pathlib.Path.glob() | Finds patterns in path names and returns a generator object |


In [ ]:
# Get .txt files with os
for f_name in os.listdir('some_directory'):
    if f_name.endswith('.txt'):
        print(f_name)

data_02.txt
data_03.txt
data_01.txt
data_02_backup.txt
data_01_backup.txt
data_03_backup.txt


In [ ]:
# with fnmatch
for file_name in os.listdir('some_directory/'):
    if fnmatch.fnmatch(file_name, '*.txt'): # only text files
        print(file_name)

print('\n')    
for filename in os.listdir('some_directory/'):
    if fnmatch.fnmatch(filename, 'data_*_backup.txt'): # only data backup files
        print(filename)

data_02.txt
data_03.txt
data_01.txt
data_02_backup.txt
data_01_backup.txt
data_03_backup.txt


data_02_backup.txt
data_01_backup.txt
data_03_backup.txt


data_02_backup.txt
data_01_backup.txt
data_03_backup.txt


In [72]:
glob.glob('*.py')

for name in glob.glob('*[0-9]*.txt'):
    print(name)
    
for file in glob.iglob('**/*.py', recursive=True):
    print(file)

some_directory/admin.py
some_directory/tests.py
some_directory/sub_dir/file2.py
some_directory/sub_dir/file1.py


In [74]:
p = Path('./some_directory')
for name in p.glob('*.p*'):
    print(name)

some_directory/admin.py
some_directory/tests.py


## Traversing Directories and Processing Files with `os.walk()`

In [75]:
# Walking a directory tree and printing the names of the directories and files
for dirpath, dirnames, files in os.walk('.'):
    print(f'Found directory: {dirpath}')
    for file_name in files:
        print(file_name)

Found directory: .
tutorial_1.ipynb
data.txt
Found directory: ./some_directory
data_02.txt
data_03.txt
data_01.txt
data_02_backup.txt
data_01_backup.txt
admin.py
tests.py
data_03_backup.txt
Found directory: ./some_directory/sub_dir
file2.py
file1.py
Found directory: ./2026
Found directory: ./2026/09
Found directory: ./2026/09/04
Found directory: ./new_dir
Found directory: ./new_dir_2


In [76]:
for dirpath, dirnames, files in os.walk('.', topdown=False):
    print(f'Found directory: {dirpath}')
    for file_name in files:
        print(file_name)

Found directory: ./some_directory/sub_dir
file2.py
file1.py
Found directory: ./some_directory
data_02.txt
data_03.txt
data_01.txt
data_02_backup.txt
data_01_backup.txt
admin.py
tests.py
data_03_backup.txt
Found directory: ./2026/09/04
Found directory: ./2026/09
Found directory: ./2026
Found directory: ./new_dir
Found directory: ./new_dir_2
Found directory: .
tutorial_1.ipynb
data.txt


## Making Temporary Files and Directories

In [ ]:
# # Create a temporary file and write some data to it
# fp = TemporaryFile('w+t')
# fp.write('Hello universe!')

# # Go back to the beginning and read data from file
# fp.seek(0)
# data = fp.read()

# # Close the file, after which it will be removed
# fp.close()

with TemporaryFile('w+t') as fp:
    fp.write('Hello universe!')
    fp.seek(0)
    fp.read()
# File is now closed and removed

In [ ]:
# For directories
with TemporaryDirectory() as tmpdir:
    print('Created temporary directory ', tmpdir)
    os.path.exists(tmpdir)

## Deleting Files and Directories with `os`, `shutil`, and `pathlib`

| Function      | Description |
| ----------- | ----------- |
| os.remove()      | Deletes a file and does not delete directories |
| os.unlink() | Is identical to os.remove() and deletes a single file |
| pathlib.Path.unlink()   | Deletes a file and cannot delete directories  |
| os.rmdir() | Deletes an empty directory |
| pathlib.Path.rmdir() | Deletes an empty directory | 
| shutil.rmtree() | Deletes entire directory tree and can be used to delete non-empty directories |


In [ ]:
data_file = './some_directory/data_03.txt'
os.remove(data_file)

# data_file = './some_directory/data_03.txt'
# os.unlink(data_file)

# # Avoid OSError
# data_file = './some_directory/data_03.txt'

# # If the file exists, delete it
# if os.path.isfile(data_file):
#     os.remove(data_file)
# else:
#     print(f'Error: {data_file} not a valid filename')

# # Use exception handling
# try:
#     os.remove(data_file)
# except OSError as e:
#     print(f'Error: {data_file} : {e.strerror}')

# Pathlib
# try:
#     data_file.unlink()
# except IsADirectoryError as e:
#     print(f'Error: {data_file} : {e.strerror}')

In [ ]:
# # And similar for directories

# trash_dir = ''
# try:
#     os.rmdir(trash_dir)
# except OSError as e:
#     print(f'Error: {trash_dir} : {e.strerror}')
    
# trash_dir = Path('')
# try:
#     trash_dir.rmdir()
# except OSError as e:
#     print(f'Error: {trash_dir} : {e.strerror}')
    
# trash_dir = ''
# try:
#     shutil.rmtree(trash_dir)
# except OSError as e:
#     print(f'Error: {trash_dir} : {e.strerror}')
    
# for dirpath, dirnames, files in os.walk('.', topdown=False):
#     try:
#         os.rmdir(dirpath)
#     except OSError as ex:
#         pass

## Copying, Moving, and Renaming Files and Directories

Use shutil.copy(src, dst); copy2() for saving metadata.

Use shutil.copytree(src, dst) for copying directories.

Similarly, shutil.move(src, dst) and os.rename(src, dst).

In [ ]:
# If using pathlib:

data_file = Path('data_01.txt')
data_file.rename('data.txt')

## Archiving (i.e. compressing via Zip, Tar, or similar)

In [ ]:
with zipfile.ZipFile('data.zip', 'r') as zipobj:
    zipobj.namelist() # to get a list of files in the archive
    bar_info = zipobj.getinfo('sub_dir/bar.py') # to retrieve info about the files in the archive
    bar_info.file_size # date_time, compress_size, filename, etc.

data_zip = zipfile.ZipFile('data.zip', 'r')
data_zip.extract('file1.py') # to extract from archive
data_zip.extractall(path='extract_dir', pwd='Quish3@o') # for files with a password

In [ ]:
import zipfile

file_list = ['file1.py', 'sub_dir/', 'sub_dir/bar.py', 'sub_dir/foo.py']
with zipfile.ZipFile('new.zip', 'w') as new_zip:
    for name in file_list:
        new_zip.write(name)
        
# Open a ZipFile object in append mode
with zipfile.ZipFile('new.zip', 'a') as new_zip:
    new_zip.write('data.txt')
    new_zip.write('latin.txt')

There is also coding for TAR files, as well as an alternative method using shutil.make_archive().

We can also read data from multiple input streams usng fileinput.input()

In [ ]:
# File: fileinput-example.py
import fileinput
import sys

files = fileinput.input()
for line in files:
    if fileinput.isfirstline():
        print(f'\n--- Reading {fileinput.filename()} ---')
    print(' -> ' + line, end='')
print()